# 00 — GPU probe
Checks CUDA, model download and throughput in the isolated Kaggle environment before long runs.

In [ ]:
import time, torch, sys
print(sys.version, torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory / 1e9, "GB")
import os; print("cpus", os.cpu_count())
import psutil; print("ram", psutil.virtual_memory().total / 1e9, "GB")

In [ ]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cuda")
m.half()
texts = [f"global healthcare private limited {i} | 807 technology apartments delhi" for i in range(50000)]
t = time.time(); e = m.encode(texts, batch_size=512, convert_to_tensor=True); torch.cuda.synchronize()
print(f"bi-encoder: {len(texts)/(time.time()-t):,.0f} texts/s", e.shape, e.dtype)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tok = AutoTokenizer.from_pretrained(name)
ce = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1).cuda().half().eval()
if torch.cuda.device_count() > 1: ce = torch.nn.DataParallel(ce)
a = texts[:40000]; b = [t[::-1] for t in a]
t = time.time()
with torch.no_grad():
    for i in range(0, len(a), 1024):
        x = tok(a[i:i+1024], b[i:i+1024], truncation=True, max_length=96, padding=True, return_tensors="pt").to("cuda")
        ce(**x)
torch.cuda.synchronize(); print(f"cross-encoder: {len(a)/(time.time()-t):,.0f} pairs/s")